# NumPy Data Type Memory & Performance Benchmark Lab
Measure how dtype choice (int8 to int64, float16 to float64) changes **memory**, **speed** and **precision** on 1 million element arrays.

## Step 1: Setup and create 1M-element arrays in 7 dtypes

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import timeit

N = 1_000_000
dtypes = [np.int8, np.int16, np.int32, np.int64,
          np.float16, np.float32, np.float64]

# Values 0..99 fit safely in every dtype, so the arrays are comparable
base = np.arange(N) % 100
arrays = {np.dtype(d).name: base.astype(d) for d in dtypes}

for name, arr in arrays.items():
    print(f"{name:8s} shape={arr.shape} dtype={arr.dtype}")

## Step 2: Record size, itemsize and nbytes

In [ ]:
rows = []
for name, arr in arrays.items():
    rows.append({"dtype": name,
                 "size": arr.size,
                 "itemsize": arr.itemsize,
                 "nbytes": arr.nbytes,
                 "MB": round(arr.nbytes / 1e6, 2)})
mem_df = pd.DataFrame(rows)
mem_df

## Step 3: Time `sum` and `multiply` for each dtype

In [ ]:
times = []
for name, arr in arrays.items():
    t_sum = timeit.timeit(lambda: arr.sum(), number=50) / 50
    t_mul = timeit.timeit(lambda: arr * 2, number=50) / 50
    times.append({"dtype": name,
                  "sum_ms": round(t_sum * 1000, 4),
                  "multiply_ms": round(t_mul * 1000, 4)})
time_df = pd.DataFrame(times)

result = mem_df.merge(time_df, on="dtype")
result

Note: `float16.sum()` can return `inf` because the true total (about 49.5 million) is larger than the float16 maximum (65504). That is itself a precision/range trade-off worth reporting.

In [ ]:
print("float16 sum:", arrays["float16"].sum())
print("float32 sum:", arrays["float32"].sum())
print("int64   sum:", arrays["int64"].sum())

## Step 4: Overflow and precision loss using `astype()`

In [ ]:
# --- Integer overflow ---
a = np.array([200]).astype(np.int8)
print("200 cast to int8 ->", a)            # wraps to -56

x = np.array([100], dtype=np.int8)
print("100 + 100 in int8 ->", x + x)       # wraps to -56

print("int8 range:", np.iinfo(np.int8))

In [ ]:
# --- Float precision loss ---
val = 0.1234567891234
print("float64:", np.float64(val))
print("float32:", np.float32(val))
print("float16:", np.float16(val))

# float16 cannot represent every integer
print("float16(2049) ->", np.float16(2049))   # 2048.0

print(np.finfo(np.float16))
print(np.finfo(np.float32))
print(np.finfo(np.float64))

## Step 5: Plot memory vs time

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(16, 4))

ax[0].bar(result["dtype"], result["MB"], color="steelblue")
ax[0].set_title("Memory (MB)")

ax[1].bar(result["dtype"], result["sum_ms"], color="seagreen")
ax[1].set_title("Sum time (ms)")

ax[2].bar(result["dtype"], result["multiply_ms"], color="darkorange")
ax[2].set_title("Multiply time (ms)")

for a_ in ax:
    a_.tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.savefig("memory_vs_time.png", dpi=150)
plt.show()

## Step 6: Recommendations
*(Edit this after you look at your own numbers.)*

- Memory scales directly with itemsize: int8 uses **8x less** memory than int64.
- Pick the **smallest dtype that safely holds your value range** (e.g. `uint8` for 0-255 pixel data).
- **float32** is usually the best balance of precision, memory and speed for ML and image work.
- Avoid int8/int16 for sums or products because of **overflow**.
- float16 saves memory but loses precision quickly and has a small range (max 65504).
- Smaller dtypes are not always faster for every operation, so always benchmark on your own data.